In [ ]:
from hardDisks import dynamicHardDisks, staticHardDisks
from molsim import HardDisks
import matplotlib.pyplot as plt
import timeit
from IPython.display import clear_output

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Exercise 3: Benchmarking HardDisk Simulation: Understanding Performance Across Python, Numba, and C++ (Pybind11)

### **Overview**
The HardDisk simulation is a computationally intensive problem, making it an ideal candidate for comparing different implementation approaches. We explore the performance of three methods:
1. **Pure Python**  
2. **Python with Numba Just-In-Time (JIT) Compilation**  
3. **Direct C++ via Pybind11**  

Each method has distinct performance characteristics due to how Python, Numba, and C++ handle computation. Below, we explain these methods and their relative performance.

---

### **The Methods**

1. **Pure Python Implementation**  
   In pure Python, the HardDisk simulation is implemented as standard Python functions. Python is an **interpreted language**, which means it executes code line-by-line at runtime. This comes at the cost of speed, particularly for computationally intensive tasks like simulations. Python also lacks native support for low-level operations such as direct memory access or CPU optimizations.

   - **Performance Bottleneck**: Python incurs overhead due to dynamic typing, interpretation, and the Global Interpreter Lock (GIL), which prevents multi-threading in most cases.

2. **Python with Numba Just-In-Time (JIT) Compilation**  
   Numba provides a simple way to significantly accelerate Python code. By using the `@numba.njit` decorator, the function is **compiled to machine code** at runtime. This eliminates interpretation overhead and allows Numba to optimize the code for CPU performance.

   - **Why Numba is Faster**:  
     - **Static Typing**: Numba infers data types at runtime, removing Python’s dynamic typing overhead.  
     - **CPU Optimization**: Numba translates Python code to machine code, enabling vectorized operations and efficient CPU usage.  
     - **Loop Optimization**: Python loops are notoriously slow, but Numba compiles them into highly optimized iterations.

   - **Result**: Near C-like performance while retaining the simplicity of Python syntax.

3. **Direct C++ via Pybind11**  
   C++ is a **compiled language** that generates highly optimized machine code during compilation. By implementing the HardDisk logic in C++ and interfacing it with Python using **Pybind11**, we achieve the best performance.

   - **Why C++ is the Fastest**:  
     - **No Interpretation**: C++ is precompiled, so the code runs directly on the CPU without any intermediate interpretation.  
     - **Memory Control**: C++ offers fine-grained control over memory allocation and access, eliminating overhead associated with Python's garbage collection.  
     - **Compiler Optimizations**: Modern C++ compilers apply powerful optimizations, such as loop unrolling, vectorization, and efficient instruction scheduling.  
     - **Low Overhead**: Pybind11 introduces minimal overhead when calling C++ functions from Python.

   - **Pybind11**: This library seamlessly connects C++ and Python, allowing C++ code to be treated like Python functions. It combines the speed of C++ with the usability of Python.

---



</div>

In [ ]:
# Simulation parameters
config = {
    "numberOfInitCycles": int(1e4),
    "numberOfProdCycles": int(1e5),
    "numberOfParticles": 20,
    "boxSize": 20.0,
    "rdfBins": 100,
    "maxDisplacement": 2.0,
    "sampleFrequency": 100,
    "periodicBoundary": False,
}

In [ ]:
# Functions to test
methods = {
    "Pure Python Dynamic": "dynamicHardDisks.py_func(**config)",
    "Numba Dynamic": "dynamicHardDisks(**config)",
    "C++ Pybind Dynamic": "hd = HardDisks(**config, runStatic=False); hd.run()",
    "Pure Python Static": "staticHardDisks.py_func(**config)",
    "Numba Static": "staticHardDisks(**config)",
    "C++ Pybind Static": "hd = HardDisks(**config, runStatic=True); hd.run()",
}

# Setup for timeit
setup_code = """
from __main__ import dynamicHardDisks, staticHardDisks, HardDisks, config
"""

# Measure runtimes automatically
runtimes = {}
for method, code in methods.items():
    print(f"Perform benchmark using {method}")
    time = timeit.timeit(stmt=code, setup=setup_code, number=25)  # 25 runs for better averaging
    avg_time = time / 25  # Average time per run
    runtimes[method] = avg_time
    clear_output()

In [ ]:
# Print the results
print("Benchmark Results:")
for method, time in runtimes.items():
    print(f"{method}: {time:.6f} seconds")

# Plotting the bar graph
plt.figure(figsize=(10, 6))
plt.bar(runtimes.keys(), runtimes.values())
plt.xlabel("Method")
plt.ylabel("Runtime (seconds)")
plt.title("Comparison of HardDisk Simulation Methods")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### **Performance Comparison**

When we benchmark the three implementations (Pure Python, Numba JIT, and C++ via Pybind11), the results are clear:

1. **Pure Python**: The slowest due to interpretation overhead, dynamic typing, and inefficient loops.
2. **Numba JIT**: Much faster because it compiles Python to machine code and optimizes the loops and CPU instructions.
3. **C++ with Pybind11**: The fastest because C++ is precompiled, highly optimized, and avoids all Python-related overhead.

### **Why the Difference in Speed?**

| **Aspect**               | **Pure Python**                 | **Numba JIT**                       | **C++ with Pybind11**             |
|--------------------------|---------------------------------|-------------------------------------|----------------------------------|
| **Code Execution**       | Interpreted at runtime          | Compiled to machine code at runtime | Precompiled to machine code      |
| **Typing**               | Dynamic (slow)                 | Static (inferred at runtime)        | Static (explicit, very fast)     |
| **Loop Efficiency**      | Slow                           | Optimized                           | Highly optimized                 |
| **Memory Management**    | High overhead (garbage collection) | Moderate (optimized by Numba)      | Direct control (fastest)         |
| **Compilation Overhead** | None                           | Low (JIT compilation at runtime)    | Done ahead of time               |
| **Optimization**         | Limited                        | Vectorization, CPU optimization     | Full compiler optimizations      |
